In [0]:
import requests
import json

url = "https://dadosabertos.camara.leg.br/api/v2/proposicoes"
params = {
    "itens": 10,
    "dataInicio": "2023-05-01",
    "dataFim": "2023-07-31",
    "keywords": "CPI",
    "ordem": "ASC",
    "ordenarPor": "id"
}

response = requests.get(url, params=params)
dados = response.json()
print(f"Total de resultados: {len(dados['dados'])}")
print(f"Links de paginacao:")
for link in dados.get('links', []):
    print(f"  {link['rel']}: {link['href']}")

print(f"\nProposicoes encontradas:")
for prop in dados['dados'][:10]:
    print(f"  {prop['siglaTipo']} {prop['numero']}/{prop['ano']} - {prop['ementa'][:150]}")

# Desafio 5 (complemento) — Proposições de CPIs (Bronze + Ouro)

## Motivo
Coleta de proposições relacionadas a CPIs para atender à Entrega 2.
Apenas 9 proposições encontradas no período de criação das 4 CPIs (mai-jul/2023).

# Coleta de Proposições Relacionadas a CPIs
Proposições encontradas via busca por keyword "CPI" no período de criação das CPIs (mai-jul/2023).

In [0]:
import requests
import time
from datetime import datetime

# Coletar todas as proposicoes com keyword CPI para o periodo completo das CPIs
proposicoes_cpi = []
data_coleta = datetime.now().isoformat()

# Periodo: quando as CPIs foram criadas (mai a jul/2023)
for mes_inicio, mes_fim in [("2023-05-01", "2023-07-31")]:
    pagina = 1
    while True:
        url = "https://dadosabertos.camara.leg.br/api/v2/proposicoes"
        params = {
            "itens": 100,
            "dataInicio": mes_inicio,
            "dataFim": mes_fim,
            "keywords": "CPI",
            "pagina": pagina,
            "ordem": "ASC",
            "ordenarPor": "id"
        }
        
        response = requests.get(url, params=params)
        if response.status_code != 200:
            break
        
        dados = response.json()
        props = dados.get('dados', [])
        if not props:
            break
        
        for p in props:
            proposicoes_cpi.append((
                p['id'],
                p.get('siglaTipo'),
                p.get('numero'),
                p.get('ano'),
                p.get('ementa'),
                p.get('dataApresentacao'),
                data_coleta
            ))
        
        links = dados.get('links', [])
        tem_proxima = any(link['rel'] == 'next' for link in links)
        if not tem_proxima:
            break
        pagina += 1
        time.sleep(0.3)

print(f"Proposicoes coletadas: {len(proposicoes_cpi)}")
for p in proposicoes_cpi[:10]:
    print(f"  {p[1]} {p[2]}/{p[3]} - {p[4][:100] if p[4] else 'Sem ementa'}")

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType

schema = StructType([
    StructField("id_proposicao", LongType(), True),
    StructField("sigla_tipo", StringType(), True),
    StructField("numero", IntegerType(), True),
    StructField("ano", IntegerType(), True),
    StructField("ementa", StringType(), True),
    StructField("data_apresentacao", StringType(), True),
    StructField("data_coleta", StringType(), True)
])

df_props = spark.createDataFrame(proposicoes_cpi, schema=schema)
df_props.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.proposicoes_cpi")
print(f"Tabela bronze.proposicoes_cpi criada com {df_props.count()} registros.")

# Entrega 2: Join CPIs × Proposições
Cruza as CPIs com os requerimentos de criação (RCP) e demais proposições relacionadas.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.cpis_proposicoes
USING DELTA
COMMENT 'Camada Ouro - Proposicoes relacionadas diretamente a cada CPI'
AS
SELECT 
    p.id_proposicao,
    p.sigla_tipo,
    p.numero,
    p.ano,
    p.ementa,
    p.data_apresentacao,
    CASE 
        WHEN LOWER(p.ementa) LIKE '%americanas%' THEN 'CPIAMERI'
        WHEN LOWER(p.ementa) LIKE '%futebol%' THEN 'CPIFUTE'
        WHEN LOWER(p.ementa) LIKE '%mst%' OR LOWER(p.ementa) LIKE '%sem terra%' THEN 'CPIMST'
        WHEN LOWER(p.ementa) LIKE '%criptomoeda%' OR LOWER(p.ementa) LIKE '%pirâmide%' THEN 'CPIPIRAM'
        WHEN LOWER(p.ementa) LIKE '%composição%' OR LOWER(p.ementa) LIKE '%regimento%' THEN 'Todas'
    END AS sigla_cpi
FROM bronze.proposicoes_cpi p
WHERE LOWER(p.ementa) LIKE '%americanas%'
   OR LOWER(p.ementa) LIKE '%futebol%'
   OR LOWER(p.ementa) LIKE '%mst%'
   OR LOWER(p.ementa) LIKE '%sem terra%'
   OR LOWER(p.ementa) LIKE '%criptomoeda%'
   OR LOWER(p.ementa) LIKE '%pirâmide%'
   OR LOWER(p.ementa) LIKE '%composição%'
   OR LOWER(p.ementa) LIKE '%regimento%'

In [0]:
%sql
SELECT 
    sigla_cpi,
    sigla_tipo,
    numero,
    ano,
    ementa
FROM ouro.cpis_proposicoes
ORDER BY sigla_cpi, ano

# Desafio 5 — Complemento: Rede de Convocados das CPIs

## Objetivo
Explorar endpoints da API para encontrar dados de convocados, depoentes e
participantes das CPIs, atendendo à Entrega 4 do desafio.

## Endpoints testados
- `/eventos/{id}/deputados` — deputados presentes no evento
- `/eventos/{id}/pauta` — pauta do evento (pode conter lista de convidados)
- `/orgaos/{id}/membros` — membros do órgão (CPI)
- `/eventos/{id}/convidados` — retornou 405 (não disponível)

## Estratégia
Buscar deputados presentes e membros das CPIs a partir dos eventos
já identificados na tabela `ouro.cpis_timeline`.

In [0]:
import requests
import json

# Testar endpoint de eventos de uma CPI especifica para ver se ha dados de convidados
url = "https://dadosabertos.camara.leg.br/api/v2/eventos/71828/deputados"
response = requests.get(url)
print(f"/eventos/71828/deputados - Status: {response.status_code}")

# Testar endpoint de pauta do evento
url2 = "https://dadosabertos.camara.leg.br/api/v2/eventos/71828/pauta"
response2 = requests.get(url2)
print(f"/eventos/71828/pauta - Status: {response2.status_code}")

# Testar endpoint de participantes de um orgao (CPI)
# Primeiro, pegar o ID de uma CPI
url3 = "https://dadosabertos.camara.leg.br/api/v2/orgaos/102846/membros"
response3 = requests.get(url3, params={"itens": 5})
print(f"/orgaos/102846/membros - Status: {response3.status_code}")

# Testar endpoint de convidados de evento
url4 = "https://dadosabertos.camara.leg.br/api/v2/eventos/71828/convidados"
response4 = requests.get(url4)
print(f"/eventos/71828/convidados - Status: {response4.status_code}")

In [0]:
import requests
import json

# 1. Deputados presentes em um evento de CPI
url = "https://dadosabertos.camara.leg.br/api/v2/eventos/71828/deputados"
response = requests.get(url)
dados = response.json()
print("=== DEPUTADOS PRESENTES NO EVENTO ===")
print(f"Total: {len(dados['dados'])}")
if dados['dados']:
    for d in dados['dados'][:5]:
        print(f"  {d.get('nome')} ({d.get('siglaPartido')})")

# Coleta de Deputados Presentes nos Eventos das CPIs
Para cada evento de CPI, buscar os deputados que participaram.

In [0]:
import requests
import time

# Carregar eventos das 4 CPIs
df_eventos_cpi = spark.sql("""
    SELECT DISTINCT sigla_cpi, id_evento, descricao_tipo, data_hora_inicio
    FROM ouro.cpis_timeline
    ORDER BY sigla_cpi, data_hora_inicio
""")
eventos_cpi = df_eventos_cpi.collect()
print(f"Total de eventos de CPIs: {len(eventos_cpi)}")

# Coletar deputados presentes em cada evento
participantes = []
erros = []
inicio = time.time()

for i, evento in enumerate(eventos_cpi, start=1):
    id_evento = evento['id_evento']
    url = f"https://dadosabertos.camara.leg.br/api/v2/eventos/{id_evento}/deputados"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        dados = response.json()
        deputados = dados.get('dados', [])
        for dep in deputados:
            participantes.append((
                evento['sigla_cpi'],
                id_evento,
                evento['descricao_tipo'],
                dep.get('id'),
                dep.get('nome'),
                dep.get('siglaPartido'),
                dep.get('siglaUf')
            ))
    else:
        erros.append(id_evento)
    
    if i % 20 == 0 or i == 1:
        decorrido = time.time() - inicio
        print(f"  {i}/{len(eventos_cpi)} eventos | {len(participantes)} participantes | {decorrido:.0f}s")

print(f"\nConcluido! {len(participantes)} participantes, {len(erros)} erros")

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType
from datetime import datetime

data_coleta = datetime.now().isoformat()
dados_part = []

for p in participantes:
    dados_part.append((
        p[0], p[1], p[2], p[3], p[4], p[5], p[6], data_coleta, 'api_camara_eventos_deputados'
    ))

schema = StructType([
    StructField("sigla_cpi", StringType(), True),
    StructField("id_evento", LongType(), True),
    StructField("descricao_tipo", StringType(), True),
    StructField("id_deputado", LongType(), True),
    StructField("nome_deputado", StringType(), True),
    StructField("sigla_partido", StringType(), True),
    StructField("sigla_uf", StringType(), True),
    StructField("data_coleta", StringType(), True),
    StructField("origem", StringType(), True)
])

df_part = spark.createDataFrame(dados_part, schema=schema)

spark.sql("DROP TABLE IF EXISTS bronze.cpis_participantes")
df_part.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.cpis_participantes")

print(f"Tabela bronze.cpis_participantes criada com {df_part.count()} registros.")

# Entrega 4: Rede de Convocados das CPIs
Deputados que participaram dos eventos de cada CPI, com contagem de presenças.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.cpis_convocados
USING DELTA
COMMENT 'Camada Ouro - Rede de participantes das CPIs'
AS
SELECT 
    sigla_cpi,
    id_deputado,
    MAX(nome_deputado) AS nome_deputado,
    MAX(sigla_partido) AS sigla_partido,
    MAX(sigla_uf) AS sigla_uf,
    COUNT(*) AS qtd_participacoes,
    COUNT(DISTINCT id_evento) AS qtd_eventos,
    COUNT(DISTINCT descricao_tipo) AS qtd_tipos_evento
FROM bronze.cpis_participantes
GROUP BY sigla_cpi, id_deputado
ORDER BY sigla_cpi, qtd_participacoes DESC

In [0]:
%sql
-- Top 5 participantes de cada CPI
SELECT 
    sigla_cpi,
    nome_deputado,
    sigla_partido,
    sigla_uf,
    qtd_participacoes
FROM ouro.cpis_convocados
WHERE sigla_cpi = 'CPIPIRAM'
ORDER BY qtd_participacoes DESC
LIMIT 5